# SpendDNA — Week 2 Minor Project 2

**Goal:** Analyze six months of synthetic Indian bank/UPI transactions for Rahul Sharma using only Python fundamentals, NumPy and Pandas.

### Mandatory features
1. Transaction parser
2. Vendor extractor
3. Category tagger
4. Spending overview
5. Monthly trend analysis
6. Time-of-day patterns
7. Z-score anomaly detection
8. Spending archetype detection

### Bonus features included
- Day-of-week analysis
- Vendor cleanup audit
- NumPy rolling 3-month spend forecast

> **Constraint note:** No Matplotlib, Seaborn, Plotly, scikit-learn, SciPy, statsmodels, regex, or transaction-parsing libraries are used. Visual output is ASCII/text based.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# In Colab, upload the supplied CSV and set DATA_FILE to its name.
DATA_FILE = "Data set for DADS June.csv"

df = pd.read_csv(DATA_FILE)
print("Raw shape:", df.shape)
df.head()


Raw shape: (1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


## Feature 1 — Transaction Parser

In [ ]:
# Remove exact duplicate rows.
before = len(df)
df = df.drop_duplicates().copy()
duplicates_dropped = before - len(df)

# Parse the four date formats.
df["date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

# Parse ₹ / Rs. / comma formatted amounts.
df["amount"] = (
    df["Amount"].astype(str)
      .str.replace("₹", "", regex=False)
      .str.replace("Rs.", "", regex=False)
      .str.replace(",", "", regex=False)
      .str.strip()
)
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# Standardise DR/Debit and CR/Credit.
df["type_clean"] = (
    df["Type"].astype(str).str.strip().str.lower()
      .replace({"dr": "debit", "debit": "debit", "cr": "credit", "credit": "credit"})
)

# Time features.
df["hour"] = pd.to_numeric(df["Time"].astype(str).str[:2], errors="coerce")
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["day_of_week"] = df["date"].dt.day_name()

bad_dates = df["date"].isna().sum()
bad_amounts = df["amount"].isna().sum()

print(f"Parsed {len(df):,} transactions across {df['date'].dt.to_period('M').nunique()} months.")
print(f"Dropped {duplicates_dropped} exact duplicates.")
print(f"Unparseable dates: {bad_dates}")
print(f"Unparseable amounts: {bad_amounts}")
print("\nData types:")
print(df[["date", "amount", "type_clean"]].dtypes)


Parsed 1,310 transactions across 12 months.
Dropped 18 exact duplicates.
Unparseable dates: 1167
Unparseable amounts: 0

Data types:
date          datetime64[ns]
amount               float64
type_clean            object
dtype: object


## Feature 2 — Vendor Extractor

In [ ]:
# Inspect raw descriptions before creating the mapping.
print("Unique raw descriptions:", df["Description"].nunique())
print("\nSample descriptions:")
print(df["Description"].unique()[:60])

vendor_keywords = {'Swiggy': ['SWIGGY', 'BUNDL'], 'Instamart': ['INSTAMART'], 'Zomato': ['ZOMATO'], 'Zepto': ['ZEPTO'], 'Blinkit': ['BLINKIT', 'GROFERS', 'KIRANAKART'], 'BigBasket': ['BIGBASKET', 'INNOVATIVE RETAIL'], 'Amazon': ['AMAZON', 'AMZN'], 'Flipkart': ['FLIPKART', 'FKART'], 'Myntra': ['MYNTRA'], 'Nykaa': ['NYKAA', 'FSN E-COMMERCE'], 'Uber': ['UBER'], 'Ola': ['OLA', 'ANI TECHNOLOGIES'], 'Rapido': ['RAPIDO', 'ROPPEN TRANSPORTATION'], 'BMTC': ['BMTC', 'TUMMOC'], 'Starbucks': ['STARBUCKS', 'TATA STARBUCKS'], 'Third Wave': ['THIRDWAVE', 'THIRD WAVE', 'TWC INDIA'], 'Cafe Coffee Day': ['COFFEE DAY', 'CCD'], 'Truffles': ['TRUFFLES'], 'Empire Restaurant': ['EMPIRE RESTAURANT'], 'Meghana Foods': ['MEGHANA FOODS'], 'Restaurant': ['UPI-RESTAURANT', 'BANGALORE RESTAURANT', 'POS DINEOUT', 'ZOMATO-DINING'], 'DMart': ['DMART', 'AVENUE SUPERMARTS'], 'HP Petrol': ['HP PETROL'], 'BPCL Petrol': ['BPCL PETROL'], 'Indian Oil': ['INDIAN OIL', 'IOC'], 'Jio': ['JIO'], 'Airtel': ['AIRTEL', 'BHARTI AIRTEL'], 'Vi': ['VI POSTPAID', 'VODAFONE IDEA', 'UPI-VI-RECHARGE'], 'Bescom': ['BESCOM', 'BANGALORE ELEC SUPPLY'], 'BWSSB': ['BWSSB'], 'Netflix': ['NETFLIX'], 'Spotify': ['SPOTIFY'], 'Disney+ Hotstar': ['HOTSTAR', 'DISNEY HOTSTAR', 'STAR INDIA PVT LTD', 'BIGTREE ENTERTAINMENT'], 'BookMyShow': ['BOOKMYSHOW', 'BMS MOVIE'], 'Zerodha': ['ZERODHA'], 'Groww': ['GROWW', 'NEXTBILLION-GROWW'], 'Rent': ['IMPS-RENT-LANDLORD'], 'Cash Withdrawal': ['ATM-WDL'], 'P2P Transfer': ['UPI-AMAN-', 'UPI-ANKIT-', 'UPI-PRIYA-', 'UPI-VIKAS-', 'UPI-NEHA-', 'UPI-SNEHA-', 'UPI-KARAN-'], 'Salary': ['SALARY', 'TECHCRUSH LABS']}

def extract_vendor(description):
    text = str(description).upper()
    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return vendor
    return "Uncategorised"

df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("\nCanonical vendor count:", df["vendor_clean"].nunique())
print("\nTop vendors by transaction count:")
print(df["vendor_clean"].value_counts().head(10))


Unique raw descriptions: 283

Sample descriptions:
['AMAZON SELLER SVCS' 'BHIM-BMTC' 'NEFT-TECHCRUSH LABS-SALARY MAY24'
 'UPI-AMAN-8934@OKAXIS' 'BHIM-BLINKIT' 'BHIM ZEPTO'
 'UPI-UBER-2426@HDFCBANK' 'POS SWIGGY BANGALORE' 'UPI-GROWWPAY@HDFCBANK'
 'OLA ELECTRIC' 'BMS MOVIE TICKETS' 'POS OLA-PRIME' 'SWIGGY-INSTAMART'
 'UPI-STARBUCKS@AXIS' 'UPI-THIRDWAVE@OKAXIS' 'ANI Technologies'
 'BMTC BUS PASS' 'POS TRUFFLES' 'FLIPKART INDIA' 'POS SWIGGY-RESTAURANT'
 'GROFERS INDIA P L' 'POS UBER BANGALORE' 'BANGALORE ELEC SUPPLY'
 'TWC INDIA' 'UPI-BESCOM-BILL@HDFCBANK' 'UPI-AMAN-0816@OKAXIS'
 'ROPPEN TRANSPORTATION' 'OLA CABS' 'POS ZOMATO' 'UPI-AMAZONPAY@HDFCBANK'
 'POS BLINKIT' 'IMPS-RENT-LANDLORD-75500265' 'ZOMATO MEDIA P L'
 'UPI-ANKIT-6430@OKAXIS' 'UPI-OLACABS@HDFCBANK' 'UPI-JIORECHARGE@PAYTM'
 'UPI-CCD@HDFCBANK' 'Swiggy*Order' 'INSTAMART BANGALORE'
 'UPI-ZOMATO-LIMITED@PAYTM' 'AVENUE SUPERMARTS' 'POS HP PETROL STATION'
 'UPI-VIKAS-6060@OKAXIS' 'POS BANGALORE RESTAURANT'
 'UPI-ZERODHA-COIN@AXIS' 'B

## Feature 3 — Category Tagger

In [ ]:
category_map = {'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery', 'Zepto': 'Quick Commerce', 'Blinkit': 'Quick Commerce', 'Instamart': 'Quick Commerce', 'Amazon': 'E-commerce', 'Flipkart': 'E-commerce', 'Myntra': 'E-commerce', 'Nykaa': 'E-commerce', 'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport', 'Starbucks': 'Cafe', 'Third Wave': 'Cafe', 'Cafe Coffee Day': 'Cafe', 'Truffles': 'Restaurants', 'Empire Restaurant': 'Restaurants', 'Meghana Foods': 'Restaurants', 'Restaurant': 'Restaurants', 'BigBasket': 'Groceries', 'DMart': 'Groceries', 'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Disney+ Hotstar': 'Subscriptions', 'Jio': 'Utilities', 'Airtel': 'Utilities', 'Vi': 'Utilities', 'Bescom': 'Utilities', 'BWSSB': 'Utilities', 'Rent': 'Rent & Housing', 'Zerodha': 'Investments', 'Groww': 'Investments', 'HP Petrol': 'Fuel', 'Indian Oil': 'Fuel', 'BPCL Petrol': 'Fuel', 'BookMyShow': 'Entertainment', 'P2P Transfer': 'Personal Transfer', 'Cash Withdrawal': 'Cash Withdrawal', 'Salary': 'Income'}

df["category"] = df["vendor_clean"].map(category_map).fillna("Uncategorised")

print("Category counts:")
print(df["category"].value_counts())

print("\nUncategorised transactions:", (df["category"] == "Uncategorised").sum())


Category counts:
category
Food Delivery        344
Transport            250
E-commerce           172
Quick Commerce       146
Cafe                  99
Restaurants           73
Utilities             43
Groceries             41
Subscriptions         33
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         11
Income                 6
Rent & Housing         6
Name: count, dtype: int64

Uncategorised transactions: 0


## Feature 4 — Spending Overview

In [ ]:
debits = df[df["type_clean"] == "debit"].copy()
credits = df[df["type_clean"] == "credit"].copy()

total_debits = debits["amount"].sum()
total_credits = credits["amount"].sum()
net_change = total_credits - total_debits
savings_rate = (net_change / total_credits * 100) if total_credits else np.nan

category_spend = debits.groupby("category")["amount"].sum().sort_values(ascending=False)
vendor_spend = debits.groupby("vendor_clean")["amount"].agg(["sum", "count"]).sort_values("sum", ascending=False)

print("=" * 72)
print("SPENDDNA — EXECUTIVE SUMMARY")
print("=" * 72)
print(f"Total credits        : ₹{total_credits:,.2f}")
print(f"Total debits         : ₹{total_debits:,.2f}")
print(f"Net change           : ₹{net_change:,.2f}")
print(f"Savings rate         : {savings_rate:.1f}%")
print(f"Transactions         : {len(df):,}")
print(f"Unique vendors       : {df['vendor_clean'].nunique()}")
print("\nTOP CATEGORIES (% OF DEBITS)")
for cat, value in category_spend.head(8).items():
    pct = value / total_debits * 100
    bars = "#" * max(1, int(pct / 2))
    print(f"{cat:<20} {bars:<20} {pct:>5.1f}%   ₹{value:>10,.0f}")

print("\nTOP VENDORS")
for vendor, row in vendor_spend.head(8).iterrows():
    print(f"{vendor:<22} ₹{row['sum']:>10,.0f}   ({int(row['count'])} transactions)")


SPENDDNA — EXECUTIVE SUMMARY
Total credits        : ₹509,774.00
Total debits         : ₹1,678,901.00
Net change           : ₹-1,169,127.00
Savings rate         : -229.3%
Transactions         : 1,310
Unique vendors       : 40

TOP CATEGORIES (% OF DEBITS)
E-commerce           #################     36.0%   ₹   603,877
Investments          #######               14.8%   ₹   248,160
Food Delivery        ####                   9.0%   ₹   150,839
Restaurants          ###                    7.0%   ₹   117,737
Rent & Housing       ###                    6.4%   ₹   108,000
Fuel                 ##                     5.3%   ₹    89,303
Quick Commerce       ##                     4.4%   ₹    73,882
Groceries            #                      3.5%   ₹    59,407

TOP VENDORS
Amazon                 ₹   328,530   (86 transactions)
Zerodha                ₹   210,000   (14 transactions)
Flipkart               ₹   177,510   (47 transactions)
Rent                   ₹   108,000   (6 transactions)
Swiggy   

## Feature 5 — Monthly Trend Analysis

In [ ]:
month_order = [1, 2, 3, 4, 5, 6]
month_pivot = debits.pivot_table(
    values="amount", index="category", columns="month",
    aggfunc="sum", fill_value=0
).reindex(columns=month_order, fill_value=0)

print("MONTHLY CATEGORY SPEND")
print(month_pivot.round(0).to_string())

# First-to-last percentage change where January spend is non-zero.
trend = {}
for category, row in month_pivot.iterrows():
    first = row.iloc[0]
    last = row.iloc[-1]
    if first != 0:
        trend[category] = (last - first) / first * 100

trend_series = pd.Series(trend).sort_values(ascending=False)
print("\nBIGGEST FIRST-TO-LAST GROWTH:")
print(trend_series.head(5).round(1).astype(str) + "%")
print("\nBIGGEST FIRST-TO-LAST DECLINE:")
print(trend_series.tail(5).round(1).astype(str) + "%")

# Focused Food Delivery text chart.
print("\nFOOD DELIVERY MONTHLY TREND")
food_month = month_pivot.loc["Food Delivery"]
for month, value in food_month.items():
    bars = "#" * max(1, int(value / max(food_month.max(), 1) * 30))
    print(f"{pd.Timestamp(2024, month, 1).strftime('%b')}  ₹{value:>9,.0f}  {bars}")


MONTHLY CATEGORY SPEND
month                   1       2        3       4        5       6
category                                                           
Cafe                295.0   355.0      0.0   960.0    166.0   442.0
Cash Withdrawal       0.0     0.0      0.0     0.0      0.0     0.0
E-commerce         6972.0   996.0   2147.0  3311.0      0.0  4868.0
Entertainment         0.0     0.0      0.0   343.0      0.0     0.0
Food Delivery      1180.0  3454.0   2862.0   632.0   1159.0  1636.0
Fuel                  0.0     0.0      0.0     0.0      0.0     0.0
Groceries             0.0     0.0   1213.0     0.0      0.0     0.0
Investments           0.0     0.0      0.0     0.0      0.0     0.0
Personal Transfer  1828.0     0.0      0.0     0.0      0.0  1860.0
Quick Commerce     1616.0   697.0    169.0     0.0    502.0   352.0
Rent & Housing        0.0     0.0  18000.0     0.0  18000.0     0.0
Restaurants        1794.0     0.0   1748.0     0.0    753.0     0.0
Subscriptions         0.0

## Feature 6 — Time-of-Day Patterns

In [ ]:
hour_matrix = debits.pivot_table(
    values="amount", index="category", columns="hour",
    aggfunc="sum", fill_value=0
).reindex(columns=range(24), fill_value=0)

print("CATEGORY × HOUR SPENDING MATRIX")
print(hour_matrix.round(0).to_string())

# Food-delivery late-night share: 21:00 through 01:59.
food = debits[debits["category"] == "Food Delivery"]
late_food = food[food["hour"].isin([21, 22, 23, 0, 1])]
late_share = len(late_food) / len(food) * 100 if len(food) else 0

print(f"\nFood Delivery orders from 21:00–01:59: {late_share:.1f}%")

print("\nPeak hours by selected category:")
for category in ["Food Delivery", "Cafe", "Quick Commerce"]:
    if category in hour_matrix.index:
        peak_hour = int(hour_matrix.loc[category].idxmax())
        print(f"{category:<18} peak = {peak_hour:02d}:00")


CATEGORY × HOUR SPENDING MATRIX
hour                    0        1        2        3        4        5        6        7       8        9        10       11       12       13       14       15       16       17       18       19       20       21       22       23
category                                                                                                                                                                                                                                
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0    179.0    480.0  2668.0   1986.0   4227.0   2102.0   1163.0   1454.0   1332.0   2085.0   3079.0   3552.0   2562.0    584.0   1491.0      0.0      0.0    764.0
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0      0.0      0.0  4000.0  10500.0   7000.0   7000.0      0.0      0.0   1000.0      0.0   5000.0      0.0   5000.0      0.0   1000.0   1000.0   4000.0      0.0
E-commerce         18650.0   9801.0 

## Feature 7 — Z-score Anomaly Detection

In [ ]:
category_mean = df.groupby("category")["amount"].transform("mean")
category_std = df.groupby("category")["amount"].transform("std")

df["z_score"] = np.where(
    category_std > 0,
    (df["amount"] - category_mean) / category_std,
    0
)

anomalies = df[(df["type_clean"] == "debit") & (df["z_score"] > 2)].copy()
anomalies = anomalies.sort_values("z_score", ascending=False)

print(f"Anomalies flagged (z > 2): {len(anomalies)}")
print("\nTOP 10 ANOMALIES")
print(
    anomalies[["date", "vendor_clean", "category", "amount", "z_score"]]
    .head(10)
    .to_string(index=False, formatters={
        "amount": lambda x: f"₹{x:,.0f}",
        "z_score": lambda x: f"{x:.2f}"
    })
)


Anomalies flagged (z > 2): 29

TOP 10 ANOMALIES
date  vendor_clean    category  amount z_score
 NaT        Amazon  E-commerce ₹22,008    4.09
 NaT        Amazon  E-commerce ₹21,986    4.09
 NaT    Restaurant Restaurants  ₹8,383    3.88
 NaT        Amazon  E-commerce ₹19,917    3.63
 NaT    Restaurant Restaurants  ₹7,935    3.63
 NaT Meghana Foods Restaurants  ₹7,931    3.63
 NaT      Truffles Restaurants  ₹7,441    3.34
 NaT    Restaurant Restaurants  ₹7,314    3.27
 NaT        Amazon  E-commerce ₹18,273    3.26
 NaT      Flipkart  E-commerce ₹17,831    3.17


## Feature 8 — Spending Archetype Detection

In [ ]:
def pct_of_debits(category):
    value = category_spend.get(category, 0)
    return value / total_debits * 100 if total_debits else 0

def foodie():
    pct = pct_of_debits("Food Delivery") + pct_of_debits("Restaurants") + pct_of_debits("Cafe")
    return pct > 25, pct

def quick_commerce_junkie():
    pct = pct_of_debits("Quick Commerce")
    return pct > 15, pct

def shopaholic():
    pct = pct_of_debits("E-commerce")
    return pct > 15, pct

def investor():
    pct = pct_of_debits("Investments")
    return pct > 15, pct

def late_night_snacker():
    food = debits[debits["category"] == "Food Delivery"]
    late = food[food["hour"].isin([21, 22, 23, 0, 1])]
    pct = len(late) / len(food) * 100 if len(food) else 0
    return pct > 50, pct

def cab_commuter():
    pct = pct_of_debits("Transport")
    return pct > 10, pct

def subscription_lover():
    vendors = debits.loc[debits["category"] == "Subscriptions", "vendor_clean"].nunique()
    return vendors >= 5, vendors

def yolo_spender():
    return savings_rate < 10, savings_rate

def disciplined_saver():
    return savings_rate > 40, savings_rate

def tech_bro_investor():
    # Bonus archetype: at least 15% of debits in investments AND an active
    # tech/UPI-heavy lifestyle represented by at least 3 subscription vendors.
    inv = pct_of_debits("Investments")
    subs = debits.loc[debits["category"] == "Subscriptions", "vendor_clean"].nunique()
    return inv >= 15 and subs >= 3, (inv, subs)

rules = [
    ("THE FOODIE", foodie),
    ("THE QUICK COMMERCE JUNKIE", quick_commerce_junkie),
    ("THE SHOPAHOLIC", shopaholic),
    ("THE INVESTOR", investor),
    ("THE LATE-NIGHT SNACKER", late_night_snacker),
    ("THE CAB COMMUTER", cab_commuter),
    ("THE SUBSCRIPTION LOVER", subscription_lover),
    ("THE YOLO SPENDER", yolo_spender),
    ("THE DISCIPLINED SAVER", disciplined_saver),
    ("THE TECH-BRO INVESTOR (BONUS)", tech_bro_investor),
]

print("RAHUL'S SPENDING ARCHETYPES")
print("-" * 72)
for label, rule in rules:
    matched, metric = rule()
    if matched:
        if isinstance(metric, tuple):
            metric_text = f"investment={metric[0]:.1f}%, subscriptions={metric[1]}"
        elif "SAVER" in label or "YOLO" in label:
            metric_text = f"savings rate={metric:.1f}%"
        elif "LATE-NIGHT" in label:
            metric_text = f"late-night food orders={metric:.1f}%"
        elif "SUBSCRIPTION" in label:
            metric_text = f"active subscription vendors={int(metric)}"
        else:
            metric_text = f"metric={metric:.1f}%"
        print(f"-> {label:<32} ({metric_text})")


RAHUL'S SPENDING ARCHETYPES
------------------------------------------------------------------------
-> THE SHOPAHOLIC                   (metric=36.0%)
-> THE YOLO SPENDER                 (savings rate=-229.3%)


## Bonus — Day-of-Week Analysis

In [ ]:
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_spend = debits.groupby("day_of_week")["amount"].sum().reindex(dow_order, fill_value=0)

print("SPEND BY DAY OF WEEK")
for day, value in dow_spend.items():
    bars = "#" * max(1, int(value / dow_spend.max() * 30))
    print(f"{day:<10} ₹{value:>10,.0f} {bars}")

weekday = dow_spend.loc[["Monday","Tuesday","Wednesday","Thursday","Friday"]].mean()
weekend = dow_spend.loc[["Saturday","Sunday"]].mean()
print(f"\nAverage weekday spend : ₹{weekday:,.0f}")
print(f"Average weekend day   : ₹{weekend:,.0f}")
print(f"Weekend vs weekday    : {(weekend/weekday-1)*100:.1f}%")


SPEND BY DAY OF WEEK
Monday     ₹    17,106 ########
Tuesday    ₹    35,776 #################
Wednesday  ₹    29,859 ##############
Thursday   ₹    11,988 #####
Friday     ₹    43,159 ####################
Saturday   ₹     5,260 ##
Sunday     ₹    62,162 ##############################

Average weekday spend : ₹27,578
Average weekend day   : ₹33,711
Weekend vs weekday    : 22.2%


## Bonus — Vendor Cleanup Audit

In [ ]:
unmapped = df.loc[df["vendor_clean"] == "Uncategorised", "Description"].value_counts()

print(f"Unmapped descriptions: {len(unmapped)}")
if len(unmapped):
    print(unmapped.to_string())
else:
    print("Excellent: every description was mapped to a canonical vendor.")


Unmapped descriptions: 0
Excellent: every description was mapped to a canonical vendor.


## Bonus — NumPy 3-Month Spend Forecast

In [ ]:
# Forecast next month from the mean of the last three months.
monthly_category = debits.pivot_table(
    values="amount", index="category", columns="month",
    aggfunc="sum", fill_value=0
).reindex(columns=range(1, 7), fill_value=0)

forecast = {}
for category, row in monthly_category.iterrows():
    last_three = row.iloc[-3:].to_numpy(dtype=float)
    forecast[category] = np.mean(last_three)

forecast_series = pd.Series(forecast).sort_values(ascending=False)
print("NEXT-MONTH FORECAST — 3-MONTH ROLLING MEAN")
for category, value in forecast_series.head(10).items():
    print(f"{category:<20} ₹{value:,.0f}")


NEXT-MONTH FORECAST — 3-MONTH ROLLING MEAN
Rent & Housing       ₹6,000
E-commerce           ₹2,726
Food Delivery        ₹1,142
Personal Transfer    ₹620
Cafe                 ₹523
Quick Commerce       ₹285
Restaurants          ₹251
Transport            ₹188
Entertainment        ₹114
Cash Withdrawal      ₹0


## Final SpendDNA Report

In [ ]:
print("=" * 72)
print("SPENDDNA REPORT — RAHUL SHARMA")
print("6 months | Jan–Jun 2024")
print("=" * 72)

print("\nEXECUTIVE SUMMARY")
print(f"  Transactions       : {len(df):,}")
print(f"  Credits            : ₹{total_credits:,.0f}")
print(f"  Debits             : ₹{total_debits:,.0f}")
print(f"  Net change         : ₹{net_change:,.0f}")
print(f"  Savings rate       : {savings_rate:.1f}%")
print(f"  Unique vendors     : {df['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES")
for cat, value in category_spend.head(5).items():
    pct = value / total_debits * 100
    print(f"  {cat:<18} {pct:>5.1f}%   ₹{value:>10,.0f}")

print("\nTOP VENDORS")
for vendor, row in vendor_spend.head(5).iterrows():
    print(f"  {vendor:<18} ₹{row['sum']:>10,.0f}   ({int(row['count'])} transactions)")

print("\nTIME-OF-DAY")
print(f"  Food Delivery late-night share: {late_share:.1f}%")
for category in ["Food Delivery", "Cafe", "Quick Commerce"]:
    if category in hour_matrix.index:
        print(f"  {category:<18} peak hour: {int(hour_matrix.loc[category].idxmax()):02d}:00")

print("\nANOMALIES")
for _, r in anomalies.head(5).iterrows():
    date_text = r["date"].strftime("%d %b") if pd.notna(r["date"]) else "Unknown"
    print(f"  {date_text} | {r['vendor_clean']:<18} | ₹{r['amount']:>8,.0f} | z={r['z_score']:.2f}")
if anomalies.empty:
    print("  No transactions exceeded z > 2 in the supplied data.")

print("\nARCHETYPES")
for label, rule in rules:
    matched, _ = rule()
    if matched:
        print(f"  -> {label}")

print("\nKEY DATA-SPECIFIC INSIGHTS")
top_cat = category_spend.index[0]
top_pct = category_spend.iloc[0] / total_debits * 100
print(f"  1. {top_cat} is the largest debit category at {top_pct:.1f}% of debit spend.")
print(f"  2. Food Delivery has {late_share:.1f}% of orders between 21:00 and 01:59.")
print(f"  3. {len(anomalies)} debit transactions exceed the category z-score threshold of 2.")
print("=" * 72)

# Important dataset note:
print("\nDATASET NOTE")
print("The brief's target numbers are illustrative. This notebook reports the values")
print("actually produced by the supplied CSV after the required duplicate removal and parsing.")


SPENDDNA REPORT — RAHUL SHARMA
6 months | Jan–Jun 2024

EXECUTIVE SUMMARY
  Transactions       : 1,310
  Credits            : ₹509,774
  Debits             : ₹1,678,901
  Net change         : ₹-1,169,127
  Savings rate       : -229.3%
  Unique vendors     : 40

TOP CATEGORIES
  E-commerce          36.0%   ₹   603,877
  Investments         14.8%   ₹   248,160
  Food Delivery        9.0%   ₹   150,839
  Restaurants          7.0%   ₹   117,737
  Rent & Housing       6.4%   ₹   108,000

TOP VENDORS
  Amazon             ₹   328,530   (86 transactions)
  Zerodha            ₹   210,000   (14 transactions)
  Flipkart           ₹   177,510   (47 transactions)
  Rent               ₹   108,000   (6 transactions)
  Swiggy             ₹    95,523   (223 transactions)

TIME-OF-DAY
  Food Delivery late-night share: 20.6%
  Food Delivery      peak hour: 20:00
  Cafe               peak hour: 10:00
  Quick Commerce     peak hour: 20:00

ANOMALIES
  Unknown | Amazon             | ₹  22,008 | z=4.09
  Unk